# ML Algorithms Lab Practical — K-Nearest Neighbors (KNN) Classifier

This is classifier **3** in the series. The hyperparameter grid follows the uploaded *Machine Learning Algorithms Cheatsheet*, **Parameter Grids** section (PDF page 5).

## Every KNN hyperparameter in the cheat-sheet grid is included

- `n_neighbors`: `range(1, 10)` → 1 through 9
- `algorithm`: `["kd_tree", "ball_tree"]`
- `weights`: `["uniform", "distance"]`
- `metric`: `["euclidean", "manhattan", "minkowski"]`
- `p`: `[1, 2]`

**Important implementation detail:** `p` directly controls Minkowski distance. For Euclidean/Manhattan metrics, scikit-learn ignores `p`; the notebook still includes the complete requested Cartesian grid so every listed value is represented. This can create redundant parameter combinations. The grid has 9 × 2 × 2 × 3 × 2 = **216 combinations per search**. Since `kd_tree`/`ball_tree` do not support every possible metric in every scikit-learn version (notably some Minkowski configurations), `GridSearchCV` uses `error_score=np.nan` to skip invalid combinations rather than aborting the whole search.

## Included workflow
EDA; robust preprocessing; scaling (essential for KNN); stratified train/test split; nested CV and GridSearchCV; PCA vs no-PCA; accuracy, balanced accuracy, macro/weighted precision-recall-F1, MCC, Cohen's kappa, log loss, ROC-AUC/AP where supported; confusion matrices; ROC and PR curves; learning curve; top grid candidates; paired t-test and Wilcoxon significance tests.

`random_state=42` is fixed for reproducibility in splitting/PCA; KNeighborsClassifier itself has no `random_state` parameter. The cheat sheet does not give KNN exact ranges beyond the grid above; no other KNN model hyperparameter has been silently substituted. This notebook has been syntax-checked, not executed against your dataset.

In [ ]:
# ============================================================
# 0. IMPORTS
# ============================================================
import os, glob, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel, wilcoxon

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    matthews_corrcoef, cohen_kappa_score, log_loss, roc_auc_score,
    average_precision_score, roc_curve, precision_recall_curve, auc
)

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE=42
np.random.seed(RANDOM_STATE)
sns.set_theme()
print("Imports complete. Random state:", RANDOM_STATE)


# ============================================================
# 1. LOAD CSV AND INSPECT
# ============================================================
csv_files=glob.glob("*.csv")
if len(csv_files)==1:
    default_path=csv_files[0]
    print("Found CSV:", default_path)
    file_path=default_path if input("Use this file? [Y/n]: ").strip().lower() in ("","y","yes") else input("Enter CSV path: ").strip()
elif len(csv_files)>1:
    print("CSV files:", csv_files)
    file_path=input("Enter CSV filename/path: ").strip()
else:
    file_path=input("Enter CSV file path: ").strip()

df=pd.read_csv(file_path)
print("Shape:",df.shape)
display(df.head())
df.info()
display(df.describe(include="all").T)
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("Duplicate rows:",df.duplicated().sum())
print("Columns:",df.columns.tolist())


# ============================================================
# 2. TARGET, CLEANING, EDA
# ============================================================
print("Available columns:",df.columns.tolist())
TARGET=input("Enter exact classification target column: ").strip()
if TARGET not in df.columns:
    raise ValueError(f"Target {TARGET!r} not found.")
df=df.drop_duplicates().reset_index(drop=True)
df=df.loc[df[TARGET].notna()].copy()
y=df[TARGET].copy()
X=df.drop(columns=[TARGET]).copy()

id_like=[]
for col in X.columns:
    name=str(col).lower()
    ratio=X[col].nunique(dropna=True)/max(len(X),1)
    if name in {"id","record_id","row_id","customer_id","employee_id","sample_id"}:
        id_like.append(col)
    elif name.endswith("_id") and ratio>0.80:
        id_like.append(col)
    elif ratio>0.98 and X[col].nunique(dropna=True)>20:
        id_like.append(col)
if id_like:
    print("Dropping likely ID columns:",id_like)
    X=X.drop(columns=id_like)

if y.nunique()<2:
    raise ValueError("Classification requires at least two classes.")
display(y.value_counts().to_frame("class_count"))
display(y.value_counts(normalize=True).to_frame("class_proportion"))

plt.figure(figsize=(8,4))
sns.countplot(x=y.astype(str),order=y.astype(str).value_counts().index)
plt.title("Target Class Distribution")
plt.xticks(rotation=35,ha="right"); plt.tight_layout(); plt.show()

num_eda=X.select_dtypes(include=np.number).columns.tolist()
cat_eda=X.select_dtypes(exclude=np.number).columns.tolist()
if num_eda:
    X[num_eda].hist(figsize=(14,9),bins=25); plt.suptitle("Numeric Feature Histograms"); plt.tight_layout(); plt.show()
    if len(num_eda)>1:
        plt.figure(figsize=(10,7)); sns.heatmap(X[num_eda].corr(),cmap="coolwarm",center=0)
        plt.title("Numeric Feature Correlations"); plt.tight_layout(); plt.show()
for col in cat_eda[:8]:
    plt.figure(figsize=(8,3)); X[col].astype(str).value_counts().head(15).plot(kind="bar")
    plt.title(f"Top categories: {col}"); plt.tight_layout(); plt.show()


# ============================================================
# 3. STRATIFIED TRAIN/TEST SPLIT
# ============================================================
if y.value_counts().min()<2:
    raise ValueError("At least one class has fewer than 2 records; stratified split is unreliable.")
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.20,random_state=RANDOM_STATE,stratify=y
)
print("Train:",X_train.shape,"Test:",X_test.shape)
display(y_train.value_counts(normalize=True).to_frame("train_proportion"))
display(y_test.value_counts(normalize=True).to_frame("test_proportion"))


# ============================================================
# 4. PREPROCESSING — IMPUTE + SCALE NUMERIC, IMPUTE + ONE-HOT CATEGORICAL
# ============================================================
num_cols=X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols=X_train.select_dtypes(exclude=np.number).columns.tolist()

def make_ohe():
    try: return OneHotEncoder(handle_unknown="ignore",sparse_output=False)
    except TypeError: return OneHotEncoder(handle_unknown="ignore",sparse=False)

num_pipe=Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())])
cat_pipe=Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("encoder",make_ohe())])
parts=[]
if num_cols: parts.append(("num",num_pipe,num_cols))
if cat_cols: parts.append(("cat",cat_pipe,cat_cols))
if not parts: raise ValueError("No usable features found.")
preprocessor=ColumnTransformer(parts,remainder="drop")

# PCA variant applies PCA only to the scaled numeric branch; categorical OHE columns pass through.
pca_parts=[]
if len(num_cols)>=2:
    pca_num=Pipeline([
        ("imputer",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler()),
        ("pca",PCA(n_components=0.95,random_state=RANDOM_STATE))
    ])
    pca_parts.append(("num",pca_num,num_cols))
elif num_cols:
    pca_parts.append(("num",num_pipe,num_cols))
if cat_cols: pca_parts.append(("cat",cat_pipe,cat_cols))
pca_preprocessor=ColumnTransformer(pca_parts,remainder="drop")
print("Numeric features:",num_cols)
print("Categorical features:",cat_cols)


# ============================================================
# 5. KNN GRID — EXACT CHEAT-SHEET PARAMETER VALUES
# ============================================================
param_grid={
    "model__n_neighbors": list(range(1,10)),
    "model__algorithm": ["kd_tree","ball_tree"],
    "model__weights": ["uniform","distance"],
    "model__metric": ["euclidean","manhattan","minkowski"],
    "model__p": [1,2]
}
print("All cheat-sheet grid values:")
for k,v in param_grid.items(): print(k,":",v)
print("Nominal combinations per search:",9*2*2*3*2,"(invalid estimator/metric combinations are skipped).")


# ============================================================
# 6. NESTED STRATIFIED CV — PCA VS NO PCA
# ============================================================
min_class=int(y_train.value_counts().min())
outer_splits=min(5,min_class)
if outer_splits<2: raise ValueError("Not enough samples per class for cross-validation.")
outer_cv=StratifiedKFold(n_splits=outer_splits,shuffle=True,random_state=RANDOM_STATE)
fold_rows=[]
fold_predictions={}

for fold,(tr_idx,va_idx) in enumerate(outer_cv.split(X_train,y_train),start=1):
    X_tr,X_va=X_train.iloc[tr_idx],X_train.iloc[va_idx]
    y_tr,y_va=y_train.iloc[tr_idx],y_train.iloc[va_idx]
    inner_splits=min(3,int(y_tr.value_counts().min()))
    if inner_splits<2:
        print("Skipping fold due to too few minority-class samples:",fold); continue
    inner_cv=StratifiedKFold(n_splits=inner_splits,shuffle=True,random_state=RANDOM_STATE)
    for rep,prep in [("Without PCA",preprocessor),("With PCA",pca_preprocessor)]:
        pipe=Pipeline([("preprocess",clone(prep)),("model",KNeighborsClassifier())])
        search=GridSearchCV(
            pipe,param_grid=param_grid,scoring="f1_macro",cv=inner_cv,n_jobs=-1,
            refit=True,return_train_score=True,error_score=np.nan
        )
        t0=time.time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            search.fit(X_tr,y_tr)
        elapsed=time.time()-t0
        if not np.isfinite(search.best_score_):
            print(f"No valid parameter combination for fold {fold} / {rep}. Check scikit-learn version and metric support.")
            continue
        pred_tr=search.predict(X_tr); pred_va=search.predict(X_va)
        row={
            "fold":fold,"representation":rep,"best_inner_cv_macro_f1":search.best_score_,
            "accuracy":accuracy_score(y_va,pred_va),
            "balanced_accuracy":balanced_accuracy_score(y_va,pred_va),
            "precision_macro":precision_score(y_va,pred_va,average="macro",zero_division=0),
            "recall_macro":recall_score(y_va,pred_va,average="macro",zero_division=0),
            "f1_macro":f1_score(y_va,pred_va,average="macro",zero_division=0),
            "f1_weighted":f1_score(y_va,pred_va,average="weighted",zero_division=0),
            "train_macro_f1":f1_score(y_tr,pred_tr,average="macro",zero_division=0),
            "fit_seconds":elapsed,"best_params":search.best_params_
        }
        fold_rows.append(row)
        fold_predictions[(fold,rep)]={"y_true":y_va.copy(),"y_pred":pd.Series(pred_va,index=y_va.index),"model":search.best_estimator_}
        print(f"\nFold {fold} | {rep}")
        print("Best parameters:",search.best_params_)
        print(f"Best inner CV macro F1={search.best_score_:.4f}; validation macro F1={row['f1_macro']:.4f}; accuracy={row['accuracy']:.4f}; time={elapsed:.1f}s")

cv_results=pd.DataFrame(fold_rows)
if cv_results.empty: raise RuntimeError("No folds completed. Check class counts and grid compatibility.")
display(cv_results)
metrics_cv=["accuracy","balanced_accuracy","precision_macro","recall_macro","f1_macro","f1_weighted","best_inner_cv_macro_f1","fit_seconds"]
display(cv_results.groupby("representation")[metrics_cv].agg(["mean","std"]))


# ============================================================
# 7. STATISTICAL SIGNIFICANCE — MATCHED FOLD SCORES
# ============================================================
wide=cv_results.pivot(index="fold",columns="representation",values="f1_macro").dropna()
if len(wide)>=2 and {"Without PCA","With PCA"}.issubset(wide.columns):
    no_pca=wide["Without PCA"].to_numpy(); yes_pca=wide["With PCA"].to_numpy()
    t_stat,t_p=ttest_rel(no_pca,yes_pca)
    try: w_stat,w_p=wilcoxon(no_pca,yes_pca,zero_method="wilcox")
    except ValueError: w_stat,w_p=np.nan,np.nan
    print(f"Paired t-test: t={t_stat:.4f}, p={t_p:.6f}")
    print(f"Wilcoxon signed-rank: statistic={w_stat}, p={w_p}")
    print("Alpha=0.05:", "significant difference detected" if t_p<0.05 else "no significant difference detected")
    print("Caution: fold-level tests have low power with only a few folds.")
else:
    print("Insufficient matched folds for significance testing.")


# ============================================================
# 8. CV VISUALIZATIONS
# ============================================================
plt.figure(figsize=(8,4)); sns.boxplot(data=cv_results,x="representation",y="f1_macro")
sns.stripplot(data=cv_results,x="representation",y="f1_macro",color="black",alpha=.6)
plt.title("Outer-Fold Macro F1: PCA vs No PCA"); plt.tight_layout(); plt.show()

plt.figure(figsize=(8,4)); sns.lineplot(data=cv_results,x="fold",y="accuracy",hue="representation",marker="o")
plt.title("Outer-Fold Accuracy"); plt.tight_layout(); plt.show()

for rep in ["Without PCA","With PCA"]:
    fs=[f for f in range(1,outer_splits+1) if (f,rep) in fold_predictions]
    if fs:
        item=fold_predictions[(fs[-1],rep)]
        labels=np.unique(y_train)
        cm=confusion_matrix(item["y_true"],item["y_pred"],labels=labels)
        ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=labels).plot(xticks_rotation=45,cmap="Blues",values_format="d")
        plt.title(f"Fold {fs[-1]} Confusion Matrix — {rep}"); plt.tight_layout(); plt.show()


# ============================================================
# 9. FINAL TUNING ON TRAIN SET, ONE-TIME TEST EVALUATION
# ============================================================
selected_rep=cv_results.groupby("representation")["f1_macro"].mean().idxmax()
selected_prep=preprocessor if selected_rep=="Without PCA" else pca_preprocessor
print("Representation selected from outer CV:",selected_rep)
final_cv=StratifiedKFold(n_splits=min(5,int(y_train.value_counts().min())),shuffle=True,random_state=RANDOM_STATE)
final_pipe=Pipeline([("preprocess",clone(selected_prep)),("model",KNeighborsClassifier())])
final_search=GridSearchCV(
    final_pipe,param_grid=param_grid,scoring="f1_macro",cv=final_cv,n_jobs=-1,
    refit=True,return_train_score=True,error_score=np.nan
)
t0=time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_search.fit(X_train,y_train)
final_seconds=time.time()-t0
if not np.isfinite(final_search.best_score_):
    raise RuntimeError("GridSearchCV found no valid KNN configuration; inspect warnings and scikit-learn metric support.")
best_model=final_search.best_estimator_
y_pred=best_model.predict(X_test)
print("\nFINAL KNN MODEL")
print("Best hyperparameters:",final_search.best_params_)
print(f"Best CV macro F1: {final_search.best_score_:.4f}")
print(f"Tuning/fit time: {final_seconds:.2f}s")
print(classification_report(y_test,y_pred,zero_division=0))


# ============================================================
# 10. COMPREHENSIVE TEST METRICS
# ============================================================
metrics={
    "accuracy":accuracy_score(y_test,y_pred),
    "balanced_accuracy":balanced_accuracy_score(y_test,y_pred),
    "precision_macro":precision_score(y_test,y_pred,average="macro",zero_division=0),
    "precision_weighted":precision_score(y_test,y_pred,average="weighted",zero_division=0),
    "recall_macro":recall_score(y_test,y_pred,average="macro",zero_division=0),
    "recall_weighted":recall_score(y_test,y_pred,average="weighted",zero_division=0),
    "f1_macro":f1_score(y_test,y_pred,average="macro",zero_division=0),
    "f1_weighted":f1_score(y_test,y_pred,average="weighted",zero_division=0),
    "matthews_corrcoef":matthews_corrcoef(y_test,y_pred),
    "cohen_kappa":cohen_kappa_score(y_test,y_pred)
}
if hasattr(best_model,"predict_proba"):
    proba=best_model.predict_proba(X_test)
    metrics["log_loss"]=log_loss(y_test,proba,labels=best_model.classes_)
    try:
        if len(best_model.classes_)==2:
            metrics["roc_auc"]=roc_auc_score(y_test,proba[:,1])
            metrics["average_precision"]=average_precision_score((y_test==best_model.classes_[1]).astype(int),proba[:,1])
        else:
            metrics["roc_auc_ovr_macro"]=roc_auc_score(y_test,proba,multi_class="ovr",average="macro",labels=best_model.classes_)
    except ValueError as exc:
        print("ROC-AUC/AP unavailable on this test split:",exc)
print("\nHeld-out test metrics:")
display(pd.DataFrame({"metric":list(metrics.keys()),"value":list(metrics.values())}).sort_values("metric"))


# ============================================================
# 11. CONFUSION MATRIX, ROC, PRECISION-RECALL
# ============================================================
labels=best_model.classes_
cm=confusion_matrix(y_test,y_pred,labels=labels)
ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=labels).plot(xticks_rotation=45,cmap="Blues",values_format="d")
plt.title("Final Test Confusion Matrix"); plt.tight_layout(); plt.show()

if hasattr(best_model,"predict_proba"):
    proba=best_model.predict_proba(X_test)
    if len(labels)==2:
        y_binary=(y_test==labels[1]).astype(int)
        fpr,tpr,_=roc_curve(y_binary,proba[:,1]); roc_auc_value=auc(fpr,tpr)
        plt.figure(figsize=(6,5)); plt.plot(fpr,tpr,label=f"AUC={roc_auc_value:.3f}"); plt.plot([0,1],[0,1],"--")
        plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.title("ROC Curve"); plt.legend(); plt.tight_layout(); plt.show()
        prec,rec,_=precision_recall_curve(y_binary,proba[:,1]); ap=average_precision_score(y_binary,proba[:,1])
        plt.figure(figsize=(6,5)); plt.plot(rec,prec,label=f"Average Precision={ap:.3f}")
        plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-Recall Curve"); plt.legend(); plt.tight_layout(); plt.show()
    else:
        y_bin=label_binarize(y_test,classes=labels)
        plt.figure(figsize=(7,5))
        for i,cls in enumerate(labels):
            if i<proba.shape[1] and 0<y_bin[:,i].sum()<len(y_bin):
                fpr,tpr,_=roc_curve(y_bin[:,i],proba[:,i]); plt.plot(fpr,tpr,label=f"{cls} AUC={auc(fpr,tpr):.3f}")
        plt.plot([0,1],[0,1],"--"); plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
        plt.title("Multiclass ROC Curves (One-vs-Rest)"); plt.legend(); plt.tight_layout(); plt.show()


# ============================================================
# 12. LEARNING CURVE + TOP GRID CANDIDATES
# ============================================================
try:
    lc_cv=StratifiedKFold(n_splits=min(5,int(y_train.value_counts().min())),shuffle=True,random_state=RANDOM_STATE)
    sizes,tr_scores,va_scores=learning_curve(
        clone(best_model),X_train,y_train,cv=lc_cv,scoring="f1_macro",
        train_sizes=np.linspace(.3,1.0,5),n_jobs=-1,shuffle=True,random_state=RANDOM_STATE
    )
    plt.figure(figsize=(7,5)); plt.plot(sizes,tr_scores.mean(axis=1),marker="o",label="Training macro F1")
    plt.plot(sizes,va_scores.mean(axis=1),marker="o",label="Validation macro F1")
    plt.xlabel("Training examples"); plt.ylabel("Macro F1"); plt.title("Learning Curve"); plt.legend(); plt.tight_layout(); plt.show()
except Exception as exc:
    print("Learning curve skipped:",exc)

cv_table=pd.DataFrame(final_search.cv_results_).sort_values("rank_test_score")
display(cv_table[["rank_test_score","mean_test_score","std_test_score","mean_train_score","params"]].head(10))
print("Final best parameters:",final_search.best_params_)
print("Final best CV macro F1:",final_search.best_score_)


# ============================================================
# 13. STATISTICAL TEST INTERPRETATION
# ============================================================
print("Paired significance tests above compare PCA vs no-PCA outer-fold macro F1.")
print("For rigorous classifier-vs-classifier comparison, use identical folds for both models.")
print("A p-value below 0.05 is evidence of a fold-score difference, not proof of universal superiority.")
